# All-turbine baseline workflow and healthy-row diagnostics
Run cells from top to bottom. Select the repository Python environment as the kernel.

**H0 (`healthy.py`) is a training-row filter, not a risk predictor.** It uses future-event labels to identify healthy historical training days; using this mask as a prediction would leak the answer. This notebook audits H0 separately and evaluates risk scorers registered in `models/registry.py`. With an empty registry, only constant and random references run. Scores are not necessarily calibrated probabilities.

Install once from the repository root: `python -m pip install -e ".[dev]"` and `python -m pip install ipykernel`.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/protocol.yaml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from inside the repository.")
sys.path.insert(0, str(ROOT / "src"))
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import precision_recall_curve
from wt_predmaint.config import load_protocol
from wt_predmaint.models.registry import REGISTRY, build
from wt_predmaint.models.base import attach_labels, labelled
from wt_predmaint.models.healthy import healthy_days
from wt_predmaint.evaluation.runner import run_baselines
from wt_predmaint.evaluation.plotting import calendar_frame, plot_risk_timeline

TARGET = "a"
MODEL_IDS = ["B3_power_curve"]  # None: all registered models; []: references only; or ["registered_id"]
STATION = 2304510  # Only the timeline view; training uses ALL turbines.
RUN_PREPARATION = False              # Use existing prepared data
FROZEN_TEST = False                  # Evaluate validation only
cfg = load_protocol()
INTERIM = ROOT / "data/interim"
OUT = ROOT / "reports/notebook_all_turbines"
OUT.mkdir(parents=True, exist_ok=True)
print("Registered risk models:", list(REGISTRY))
print("Splits:", cfg["splits"])
print("Prediction horizon (days):", cfg["labels"]["positive_window_days"])


## 1. Data processing
Raw files must already exist in `data/raw/`. No downloading happens here. Preparation uses all available turbines and the existing scripts/configuration. Annual conversion skips existing files; subsequent scripts replace generated outputs. Raw files are preserved. The scripts currently scan through 2026; inspect the configured splits above if your intended scope ends in 2025.

In [ ]:
steps = ["build_scada_parquet.py", "build_alarm_table.py", "build_clean_data.py", "build_labels.py", "build_features.py"]
if RUN_PREPARATION:
    for script in steps:
        print("Running", script, flush=True)
        subprocess.run([sys.executable, str(ROOT / "scripts" / script)], cwd=ROOT, check=True)
else:
    print("Using cached prepared data. Set RUN_PREPARATION=True to build it.")


In [ ]:
required = [INTERIM / "features/daily_features.parquet", INTERIM / f"labels/{TARGET}_labels.parquet", INTERIM / f"labels/{TARGET}_episodes.parquet"]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise RuntimeError("Missing prepared files; run preparation: " + ", ".join(missing))
features, labels, episodes = [pd.read_parquet(p) for p in required]
table = attach_labels(features, labels, cfg)
coverage = table.assign(year=table.Day.dt.year).groupby(["year", "split"]).agg(turbines=("StationId", "nunique"), feature_days=("Day", "size"), labelled_days=("label", "count"))
display(coverage)
display(pd.crosstab(table["split"], table["status"]))
coverage.to_csv(OUT / "coverage.csv")
coverage.reset_index().plot.bar(x="year", y=["feature_days", "labelled_days"], figsize=(12, 3), title="Available all-turbine data")
plt.show()


## 2. Healthy-row mask (H0)
These are selection diagnostics, not predictive performance. Recovery windows, quality flags and train-only eligibility come from `healthy.py` and the protocol. Never interpret healthy=False as a predicted fault.

For optional ten-minute counts, run `python scripts/report_healthy_rows.py`; it reads the larger clean SCADA tables.

In [ ]:
mask = healthy_days(labels, features, cfg, TARGET)
healthy_view = features[["StationId", "Day"]].copy()
healthy_view["healthy"] = mask.to_numpy()
healthy_view["year"] = healthy_view.Day.dt.year
healthy_counts = healthy_view.groupby(["StationId", "year"]).agg(available_days=("healthy", "size"), healthy_days=("healthy", "sum"))
healthy_counts["share_healthy"] = healthy_counts.healthy_days / healthy_counts.available_days
display(healthy_counts)
healthy_counts.to_csv(OUT / f"healthy_counts_{TARGET}.csv")
healthy_counts["share_healthy"].unstack("year").plot.bar(figsize=(15, 4), ylabel="Share retained", title="H0: healthy training days by turbine/year")
plt.tight_layout()
plt.show()


## 3. Training and validation
Switch models using `MODEL_IDS` above. To add one, implement `models.base.Baseline`, import/register its constructor in `models/registry.py`, then restart the kernel and run again. Constant and random reference models are always included. H0-based normal-behaviour models must explicitly use the healthy mask in their own implementation; the runner does not apply it to every classifier.

Fitting uses labelled training rows; validation chooses thresholds under the configured false-alarm budget. Score-contract checks verify label blindness and batch independence. Missing validation years must be prepared, not replaced with training/test data.

In [ ]:
splits = ("val", "test") if FROZEN_TEST else ("val",)
for split in ("train", *splits):
    if labelled(table, split).empty:
        raise RuntimeError(f"No labelled {split} rows. Required years: {cfg['splits'][split + '_years']}. Inspect coverage above and prepare missing data.")
result = run_baselines(build(MODEL_IDS, TARGET), table, episodes, cfg, TARGET, splits=splits, frozen=FROZEN_TEST)
display(result.table)
result.table.to_csv(OUT / f"metrics_{TARGET}.csv", index=False)
for (variant, model, split), scores in result.scores.items():
    scores.to_csv(OUT / f"scores_{TARGET}_{variant}_{model}_{split}.csv", index=False)


## 4. Evaluation and final results
`pr_auc` is average precision (higher is better). `recall` is the fraction of evaluable fault episodes detected, not row-level recall. Lead time measures the earliest warning before a detected episode. False alarms count negative alert **days**, normalized by evaluated turbine-months. Read recall and false alarms together and compare with random references. Confidence bounds bootstrap turbines. Test results appear only when `FROZEN_TEST=True`; keep it false during development.

In [ ]:
metrics = ["pr_auc", "recall", "false_alarms_per_turbine_month", "median_lead_days"]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for metric, ax in zip(metrics, axes.flat):
    result.table.pivot(index="model", columns="split", values=metric).plot.bar(ax=ax, title=metric, rot=30)
fig.tight_layout()
fig.savefig(OUT / f"metric_comparison_{TARGET}.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
for (variant, model, split), scores in result.scores.items():
    if split != "val":
        continue
    frame = labelled(table, split).merge(scores, on=["StationId", "Day"], how="inner")
    if frame.label.nunique() < 2:
        print("PR curve needs both classes:", model)
        continue
    precision, recall, _ = precision_recall_curve(frame.label, frame.score)
    ax.plot(recall, precision, label=model)
ax.set(xlabel="Row-level recall", ylabel="Precision", title="Validation precision–recall curves (row-level)")
ax.legend()
fig.savefig(OUT / f"precision_recall_{TARGET}.png", dpi=150)
plt.show()


## 5. Risk timeline and data splits
Select any evaluated model below. The display turbine does not restrict training. Each line includes every feature day scored for the evaluated splits, including unlabelled days; only labelled days enter metrics. Training years have no risk line here because this view uses the runner's validation/test scores. Gaps outside evaluated splits do not prove missing raw data. The status and split bands explain the calendar.

In [ ]:
available_models = result.table.model.unique().tolist()
print("Choose PLOT_MODEL from:", available_models)
PLOT_MODEL = "B3_power_curve"
selected = [s for (_, model, _), s in result.scores.items() if model == PLOT_MODEL]
values = pd.concat(selected, ignore_index=True)
station = STATION if STATION in table.StationId.unique() else int(table.StationId.min())
cal = calendar_frame(values, labels, station, "score", str(table.Day.min().date()), str(table.Day.max().date()))
fig = plot_risk_timeline(cal, episodes, station, ylabel="Risk score (not necessarily probability)", title=f"Turbine {station}: {PLOT_MODEL}, target {TARGET.upper()}")
threshold = result.table.loc[result.table.model == PLOT_MODEL, "threshold"].iloc[0]
if pd.notna(threshold) and abs(threshold) != float("inf"):
    fig.axes[0].axhline(threshold, color="orange", linestyle="--", label="Validation threshold")
    fig.axes[0].legend()
fig.savefig(OUT / f"timeline_{TARGET}_{station}_{PLOT_MODEL}.png", dpi=150)
plt.show()
print("Saved tables and figures to", OUT)
